# Credit Card Fraud Analytics — Data Cleaning & Preparation

## Phase 2: Data Cleaning & Preparation

The objective of this phase is to investigate data-quality issues and make evidence-based cleaning decisions before exploratory analysis.

**Important:** We do not automatically remove unusual records. Each potential issue is investigated first, then a cleaning decision is made.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Load Dataset

In [ ]:
DATA_PATH = '../data/creditcard.csv'

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')

## 3. Create a Working Copy

In [ ]:
df_clean = df.copy()

print('Working copy created successfully.')

## 4. Validate Data Types

In [ ]:
dtype_summary = pd.DataFrame({
    'column': df_clean.columns,
    'dtype': df_clean.dtypes.astype(str).values
})

display(dtype_summary)

### Interpretation

The dataset consists of numeric variables. `Class` is a binary target variable with values 0 and 1.

## 5. Check Missing Values

In [ ]:
missing_summary = (
    df_clean.isna().sum()
    .to_frame('missing_count')
)

missing_summary['missing_percentage'] = (
    missing_summary['missing_count'] / len(df_clean) * 100
)

missing_summary = missing_summary[
    missing_summary['missing_count'] > 0
]

if missing_summary.empty:
    print('No missing values found.')
else:
    display(missing_summary)

### Cleaning Decision

No missing-value treatment is required if the result above is empty.

## 6. Investigate Duplicate Rows

In [ ]:
duplicate_count = df_clean.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_clean) * 100

print(f'Duplicate rows: {duplicate_count:,}')
print(f'Duplicate percentage: {duplicate_percentage:.4f}%')

### 6.1 Inspect Duplicate Records

In [ ]:
duplicates = df_clean[
    df_clean.duplicated(keep=False)
].sort_values(by=df_clean.columns.tolist())

display(duplicates.head(20))

`keep=False` marks every member of a duplicated group as a duplicate, not only the later occurrence.

### 6.2 Duplicate Class Distribution

In [ ]:
duplicate_rows = df_clean[df_clean.duplicated(keep=False)]

duplicate_class_distribution = (
    duplicate_rows['Class']
    .value_counts()
    .rename(index={0: 'Normal', 1: 'Fraud'})
)

display(duplicate_class_distribution.to_frame('count'))

In [ ]:
duplicate_class_percentage = (
    duplicate_rows['Class']
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={0: 'Normal', 1: 'Fraud'})
)

display(duplicate_class_percentage.to_frame('percentage'))

### 6.3 Check for Conflicting Duplicate Labels

In [ ]:
feature_columns = [
    col for col in df_clean.columns
    if col != 'Class'
]

duplicate_label_counts = (
    df_clean.groupby(feature_columns, dropna=False)['Class']
    .nunique()
)

conflicting_duplicate_groups = duplicate_label_counts[
    duplicate_label_counts > 1
]

print(
    'Duplicate feature groups with conflicting Class labels:',
    len(conflicting_duplicate_groups)
)

### Cleaning Decision — Duplicates

If there are no conflicting labels among exact duplicate feature groups, exact duplicate rows can be removed for a transaction-level analytical dataset.

**Do not execute the removal until the result above has been reviewed.**

In [ ]:
# Remove exact duplicate rows after reviewing the duplicate analysis above.
df_clean = df_clean.drop_duplicates().copy()

print('Shape after removing exact duplicates:', df_clean.shape)
print('Remaining duplicates:', df_clean.duplicated().sum())

**Note:** In this project, exact duplicates are removed after investigation. We do not remove near-duplicates or records that merely have similar transaction values.

## 7. Investigate Zero-Amount Transactions

In [ ]:
zero_amount_count = (df_clean['Amount'] == 0).sum()
zero_amount_percentage = (df_clean['Amount'] == 0).mean() * 100

print(f'Zero-amount transactions: {zero_amount_count:,}')
print(f'Zero-amount percentage: {zero_amount_percentage:.4f}%')

### 7.1 Zero-Amount Transactions by Class

In [ ]:
zero_amount_by_class = (
    df_clean.loc[df_clean['Amount'] == 0, 'Class']
    .value_counts()
    .rename(index={0: 'Normal', 1: 'Fraud'})
)

display(zero_amount_by_class.to_frame('count'))

In [ ]:
zero_amount_fraud_rate = (
    df_clean.loc[df_clean['Amount'] == 0, 'Class'].mean() * 100
)

overall_fraud_rate = df_clean['Class'].mean() * 100

print(f'Fraud rate among zero-amount transactions: {zero_amount_fraud_rate:.4f}%')
print(f'Overall fraud rate: {overall_fraud_rate:.4f}%')

### Cleaning Decision — Zero Amount

Zero-amount transactions are **kept**. A zero amount is not automatically a data error and may represent a meaningful transaction state. Their relationship with fraud will be analyzed later in EDA.

## 8. Check for Negative Amounts

In [ ]:
negative_amount_count = (df_clean['Amount'] < 0).sum()

print(f'Negative amount transactions: {negative_amount_count:,}')

Negative transaction amounts are not automatically invalid in every financial dataset because they can represent refunds, reversals or adjustments. If any are found, they should be investigated before removal.

## 9. Validate Class Values

In [ ]:
valid_classes = {0, 1}

invalid_classes = df_clean.loc[
    ~df_clean['Class'].isin(valid_classes),
    'Class'
].unique()

print('Invalid Class values:', invalid_classes)

### Cleaning Decision

Only `0` and `1` are valid target labels. Any other value would require investigation before modeling.

## 10. Validate Time Values

In [ ]:
print('Minimum Time:', df_clean['Time'].min())
print('Maximum Time:', df_clean['Time'].max())
print('Negative Time values:', (df_clean['Time'] < 0).sum())
print('Missing Time values:', df_clean['Time'].isna().sum())

`Time` represents elapsed seconds from the beginning of the observation period. Negative values would be structurally invalid.

## 11. Validate V1–V28 Features

In [ ]:
v_features = [f'V{i}' for i in range(1, 29)]

missing_v_features = [
    col for col in v_features
    if col not in df_clean.columns
]

print('Missing V features:', missing_v_features)

In [ ]:
infinite_counts = np.isinf(df_clean[v_features]).sum()

print('Infinite values across V1–V28:')
display(infinite_counts[infinite_counts > 0])

The V1–V28 columns are anonymized PCA-transformed features. We validate their structure and numeric values, but we do not assign unsupported business meanings to them.

## 12. Data Quality Summary After Cleaning

In [ ]:
quality_summary = pd.DataFrame({
    'check': [
        'Missing values',
        'Duplicate rows',
        'Zero-amount transactions',
        'Negative amounts',
        'Invalid Class values',
        'Negative Time values',
        'Infinite V features'
    ],
    'count': [
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        int((df_clean['Amount'] == 0).sum()),
        int((df_clean['Amount'] < 0).sum()),
        int(len(invalid_classes)),
        int((df_clean['Time'] < 0).sum()),
        int(np.isinf(df_clean[v_features]).sum().sum())
    ]
})

display(quality_summary)

## 13. Final Validation

In [ ]:
print(f'Final dataset shape: {df_clean.shape}')
print(f'Total missing values: {df_clean.isna().sum().sum():,}')
print(f'Remaining duplicate rows: {df_clean.duplicated().sum():,}')
print(f'Invalid Class values: {len(invalid_classes):,}')

## 14. Save Clean Dataset

The cleaned dataset is saved as a separate file. The original raw dataset remains unchanged.

In [ ]:
OUTPUT_PATH = '../data/creditcard_clean.csv'

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f'Clean dataset saved to: {OUTPUT_PATH}')

## 15. Phase 2 Conclusions

The dataset has been systematically checked for missing values, exact duplicates, zero-amount transactions, invalid labels, invalid time values and infinite feature values.

Exact duplicate rows are removed after investigation. Zero-amount transactions are retained because they are not automatically invalid and may provide analytical signal.

The resulting dataset is saved separately from the raw data so the original source remains unchanged.

### Next phase

**Phase 3 — Exploratory Data Analysis (EDA)** will investigate fraud patterns using transaction amount, time, class distribution and the anonymized V1–V28 features.